# VBT PRO Capability Dump And Repo Fit

Purpose:
- verify the installed `vectorbtpro` runtime
- dump the core public capability surfaces we care about
- keep fast-access source lookups for splitters, patterns, labels, execution, params, and portfolio tools
- map those capabilities back to the current repo architecture and Blackwell lane

Primary repo note:
- `/home/benji/trade_scanner2/reports/strategy_factory/vbtpro_repo_blackwell_encyclopedia_20260408.yaml`


In [ ]:
from pathlib import Path
import inspect
import json
import vectorbtpro as vbt

print('version', vbt.__version__)
print('module', vbt.__file__)


In [ ]:
key_symbols = [
    'Splitter', 'SplitterCV', 'split', 'cv_split',
    'chunked', 'Param', 'parameterized',
    'PortfolioOptimizer', 'PATSIM'
]

for name in key_symbols:
    obj = getattr(vbt, name, None)
    print(name, bool(obj), obj)
    if obj is not None:
        try:
            print('  file:', inspect.getsourcefile(obj))
        except Exception as exc:
            print('  sourcefile_error:', type(exc).__name__)


In [ ]:
REPO_ROOT = Path('/home/benji/trade_scanner2')
VBT_SRC = Path('/home/benji/Downloads/vectorbt.pro-main/vectorbtpro')
VBT_SITE = Path('/home/benji/miniconda3/envs/vectorbtpro/lib/python3.12/site-packages/vectorbtpro')

print('repo_root', REPO_ROOT)
print('vbt_src_exists', VBT_SRC.exists())
print('vbt_site_exists', VBT_SITE.exists())


In [ ]:
repo_fit_anchors = {
    'split_contracts': [
        REPO_ROOT / 'trade_scanner/strategy_factory/validation_segments.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/splits.py',
        REPO_ROOT / 'reports/strategy_factory/lane_command_contract_20260330.yaml',
    ],
    'meta_model_path': [
        REPO_ROOT / 'trade_scanner/strategy_factory/models.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/walk_forward.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/pipeline.py',
    ],
    'blackwell_lane': [
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/runner.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/gpu_executor.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/feature_builders_gpu.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/ranking.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/blackwell_ensemble/evaluation.py',
    ],
    'current_vbt_runtime': [
        REPO_ROOT / 'trade_scanner/strategy_factory/vbt_equities.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/vbtpro_native_runtime.py',
        REPO_ROOT / 'trade_scanner/strategy_factory/vbt_metadata.py',
    ],
}

for group, paths in repo_fit_anchors.items():
    print('\n##', group)
    for path in paths:
        print(path)


## Splitters And CV

Best repo fit:
- derive VBT splitters from existing repo role/date contracts
- keep `validation_segments.py` and `blackwell_ensemble/splits.py` as authority
- use VBT for adapter objects, parity views, reusable masks, and walkforward/CV iteration

Strong surfaces:
- `vbt.Splitter`
- `vbt.SplitterCV`
- `vbt.split`
- `vbt.cv_split`
- `Splitter.from_rolling`
- `Splitter.from_purged_walkforward`
- `Splitter.from_purged_kfold`


In [ ]:
splitter_source_targets = [
    VBT_SITE / 'generic/splitting/base.py',
    VBT_SITE / 'generic/splitting/sklearn_.py',
    VBT_SITE / 'generic/splitting/decorators.py',
    VBT_SITE / 'generic/splitting/purged.py',
]

needles = [
    'class Splitter',
    'def from_rolling',
    'def from_purged_walkforward',
    'def from_purged_kfold',
    'class SplitterCV',
    'def split(',
    'def cv_split(',
]

for path in splitter_source_targets:
    print('\nFILE', path)
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines, 1):
        if any(n in line for n in needles):
            print(f'{i}: {line.strip()}')


## Pattern Features And Labels

Best repo fit:
- Blackwell feature panel augmentation in `feature_builders_gpu.py`
- Lane 2 regime feature augmentation in `semantic_lane2_coarse_medium_feature_discovery.py`
- auxiliary targets only in `labels.py`

Good feature candidates:
- rolling pattern similarity scores
- pattern occurrence / distance-to-last-hit features
- trend/pivot state features
- future-statistic targets like `FMEAN`, `FSTD`, `FMAX`, `FMIN`

Leakage rule:
- pattern features must be trailing only
- look-ahead label generators belong only in label/target generation


In [ ]:
pattern_targets = [
    VBT_SITE / 'generic/accessors.py',
    VBT_SITE / 'generic/ranges.py',
    VBT_SITE / 'indicators/custom/patsim.py',
    VBT_SITE / 'labels/generators/trendlb.py',
    VBT_SITE / 'labels/generators/pivotlb.py',
    VBT_SITE / 'labels/generators/fmean.py',
]
needles = [
    'rolling_pattern_similarity',
    'find_pattern',
    'fit_pattern',
    'class PatternRanges',
    'def from_pattern_search',
    'PATSIM',
    'TRENDLB',
    'PIVOTLB',
    'FMEAN',
]

for path in pattern_targets:
    print('\nFILE', path)
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines, 1):
        if any(n in line for n in needles):
            print(f'{i}: {line.strip()}')


## Execution, Parameterization, Chunking, Caching, Jitting

Best repo fit:
- CPU-side ranking/evaluation sweeps
- split-wise diagnostics and parity views
- bounded parameter grids around existing strategy or Blackwell outputs

Do not confuse this with GPU training acceleration.
Blackwell GPU model fitting should stay in the current `gpu_executor.py` lane.


In [ ]:
speed_targets = [
    VBT_SITE / 'utils/execution.py',
    VBT_SITE / 'utils/params.py',
    VBT_SITE / 'utils/chunking.py',
    VBT_SITE / 'utils/caching.py',
    VBT_SITE / 'utils/jitting.py',
    VBT_SITE / '_settings.py',
]
needles = [
    'class ExecutionEngine',
    'def execute(',
    'class Param',
    'class Parameterizer',
    'def parameterized(',
    'class Chunker',
    'def chunked(',
    'class Cacheable',
    'def jitted(',
    'execution =',
    'chunking =',
]

for path in speed_targets:
    print('\nFILE', path)
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines, 1):
        if any(n in line for n in needles):
            print(f'{i}: {line.strip()}')


## Portfolio Simulation And Optimizer Surfaces

Best repo fit:
- research-only portfolio mirrors for Blackwell holdout scores
- alternate allocation experiments over ranked names
- keep current `ranking.py` and `evaluation.py` as authority until parity is proven

Good first use:
- `Portfolio.from_signals`
- `Portfolio.from_order_func`
- `Portfolio.from_optimizer`
- `PortfolioOptimizer`


In [ ]:
portfolio_targets = [
    VBT_SITE / 'portfolio/base.py',
    VBT_SITE / 'portfolio/pfopt/base.py',
]
needles = [
    'class Portfolio(',
    'def from_signals(',
    'def from_optimizer(',
    'def from_order_func(',
    'class PortfolioOptimizer',
    'def from_optimize_func(',
]

for path in portfolio_targets:
    print('\nFILE', path)
    lines = path.read_text().splitlines()
    for i, line in enumerate(lines, 1):
        if any(n in line for n in needles):
            print(f'{i}: {line.strip()}')


## Blackwell-Specific Guardrails

Keep:
- current GPU topology
- current split authority
- current holdout gate
- current model training stack in `gpu_executor.py`

Good VBT uses around Blackwell:
- split adapter and diagnostics
- CPU-side pattern feature blocks
- portfolio mirrors on holdout predictions
- parameterized ranking/portfolio sweeps

Anti-patterns:
- replacing GPU training with VBT PRO
- letting VBT define authoritative folds independently
- mixing VBT multiprocessing into GPU worker ownership without a topology plan


## Next Prototype Order

1. Splitter adapter over existing repo contracts.
2. Optional Blackwell pattern feature block.
3. VBT portfolio mirror over Blackwell holdout scores.
4. Parameterized CPU-side ranking/evaluation sweeps.
5. Only then consider a broader VBT-native research lane.
